# 05 — Grid Capacity Consolidation

Load i-DE, Endesa, and Viesgo capacity datasets. Standardize into unified format: `substation_id`, `latitude`, `longitude`, `available_capacity_mw`, `distributor_network`. Save consolidated grid dataset.

## Data Inputs
- `data/raw/grid_capacity/ide_iberdrola/` — i-DE capacity files
- `data/raw/grid_capacity/endesa/` — Endesa capacity files
- `data/raw/grid_capacity/viesgo/` — Viesgo capacity files
- `data/raw/additional/ree_grid/` — REE transmission data

## Data Outputs
- `data/processed/grid_consolidated.csv` — unified substation dataset
  Columns: substation_id, latitude, longitude, available_capacity_mw, distributor_network

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from src.grid_analysis import classify_grid_status, is_friction_point

PROCESSED = Path('data/processed')

# Load the unified grid capacity (already cleaned in Notebook 01)
grid = pd.read_csv(PROCESSED / 'grid_capacity_unified.csv')
print(f"Total substation records: {grid.shape[0]}")
print(f"Columns: {list(grid.columns)}")
print(f"\nBy distributor:")
print(grid['distributor_network'].value_counts())
grid.head()

## Aggregate by Substation

Some substations appear multiple times (different voltage levels or measurement periods). Aggregate to one row per substation, taking the maximum available capacity.

In [ ]:
# Check for duplicate substations
print(f"Unique substation IDs: {grid['substation_id'].nunique()}")
print(f"Total rows: {len(grid)}")

# Aggregate: max capacity per substation (across voltage levels)
# Keep first occurrence of location info
agg_funcs = {
    'substation_name': 'first',
    'provincia': 'first',
    'municipio': 'first',
    'latitude': 'first',
    'longitude': 'first',
    'available_capacity_mw': 'max',
    'distributor_network': 'first',
}

# Only aggregate voltage if it exists
if 'voltage_kv' in grid.columns:
    agg_funcs['voltage_kv'] = 'max'

grid_agg = grid.groupby('substation_id').agg(agg_funcs).reset_index()

print(f"\nAfter aggregation: {len(grid_agg)} unique substations")
print(f"Capacity stats (MW):")
print(grid_agg['available_capacity_mw'].describe().round(2))

## Classify Grid Status

Apply `classify_grid_status()` from `src/grid_analysis.py`:
- **Sufficient**: ≥ 5 MW
- **Moderate**: 1–5 MW  
- **Congested**: < 1 MW

In [ ]:
# Classify each substation
grid_agg['grid_status'] = grid_agg['available_capacity_mw'].apply(classify_grid_status)
grid_agg['is_friction'] = grid_agg['grid_status'].apply(is_friction_point)

print("Grid status distribution:")
print(grid_agg['grid_status'].value_counts())
print(f"\nFriction points (Moderate + Congested): {grid_agg['is_friction'].sum()}")

# By distributor
print("\nGrid status by distributor:")
print(pd.crosstab(grid_agg['distributor_network'], grid_agg['grid_status'], margins=True))

## Create GeoDataFrame & Save

In [ ]:
# Drop rows without valid coordinates
valid = grid_agg.dropna(subset=['latitude', 'longitude'])
print(f"Substations with valid coords: {len(valid)} / {len(grid_agg)}")

# Create GeoDataFrame
grid_gdf = gpd.GeoDataFrame(
    valid,
    geometry=gpd.points_from_xy(valid['longitude'], valid['latitude']),
    crs='EPSG:4326'
)

# Sanity check: coordinates in Spain bbox
in_spain = (
    (grid_gdf.geometry.y >= 35) & (grid_gdf.geometry.y <= 44) &
    (grid_gdf.geometry.x >= -10) & (grid_gdf.geometry.x <= 5)
)
print(f"Coords in Spain bbox: {in_spain.sum()} / {len(grid_gdf)}")
if (~in_spain).any():
    print(f"WARNING: {(~in_spain).sum()} substations outside Spain — dropping them")
    grid_gdf = grid_gdf[in_spain]

# Summary
print(f"\n{'='*60}")
print("GRID CAPACITY CONSOLIDATION — SUMMARY")
print(f"{'='*60}")
print(f"Total substations: {len(grid_gdf)}")
print(f"Sufficient: {(grid_gdf['grid_status']=='Sufficient').sum()}")
print(f"Moderate: {(grid_gdf['grid_status']=='Moderate').sum()}")
print(f"Congested: {(grid_gdf['grid_status']=='Congested').sum()}")
print(f"\nCapacity (MW): mean={grid_gdf['available_capacity_mw'].mean():.1f}, "
      f"median={grid_gdf['available_capacity_mw'].median():.1f}")

# Save GeoJSON
out_geo = PROCESSED / 'grid_substations.geojson'
grid_gdf.to_file(out_geo, driver='GeoJSON')
print(f"\nSaved: {out_geo} ({len(grid_gdf)} substations)")

# Also save as CSV (without geometry) for easy downstream use
out_csv = PROCESSED / 'grid_consolidated.csv'
grid_gdf.drop(columns='geometry').to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")